# Final-stage Chain-of-Thought Reranker (targets the #1 error mechanism)

**Motivation (from the error analysis, §8a / `error_analysis_ensemble.md`):** the binding constraint
is the reranker's top-10 precision, and the dominant mechanism is **LLM false negatives** — the
zero-shot single-pass Qwen judge (`llm_yesno`) says "no" to genuinely eligible trials and buries them
(eligible-surfaced +5.78 vs eligible-buried −11.51). A **reasoning (chain-of-thought) judge** that
works through the eligibility criteria step-by-step before answering should recover many of these.

**Design:** re-rank the frozen ensemble's **top-K** per topic with an open reasoning LLM. Final score =
`alpha * z(cot_score) + (1-alpha) * z(ensemble_rank_score)`; below K, keep the ensemble order.

## ANTI-GAMING PROTOCOL (read before running)
- The current ensemble (0.6105 on TREC22) is a **frozen, pre-registered baseline**.
- **All development — K, alpha, the prompt, the model choice — is done on TREC 2021 CV ONLY.**
- **TREC 2022 is touched exactly once**, at the end, with the config fixed on 2021. No iterating on 2022.
- Report per-topic deltas + a paired bootstrap (baseline vs +CoT) on whichever split.
- Rationale: the error analysis was performed on TREC22, so any TREC22-tuned fix would be a 9th round of
  test-adaptation. Developing on 2021 keeps the improvement independent of the test.

**Open-weight** judge (Qwen2.5-7B or a distilled reasoning model) — no closed model in the path.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q lightgbm pytrec_eval rank-bm25 sentence-transformers datasets transformers accelerate tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DATA_ROOT       = '/content/drive/MyDrive/ct_data23'
EVAL_ROOT       = f'{DATA_ROOT}/evaluation'
TREC_ROOT       = f'{EVAL_ROOT}/trec_data'
KZ_ROOT         = f'{EVAL_ROOT}/kz_data'
FULLTEXT_CORPUS = f'{DATA_ROOT}/doc_texts_fulltext.txt'
FEATURE_CACHE   = f'{DATA_ROOT}/ensemble_features_full.npz'
META_CACHE      = f'{DATA_ROOT}/ensemble_features_full_meta.json'
ENSEMBLE_MODEL  = f'{DATA_ROOT}/ensemble_full_v1.txt'   # the frozen LambdaMART booster
COT_OUT         = f'{DATA_ROOT}/cot_scores.jsonl'
LLM_MODEL       = 'Qwen/Qwen2.5-7B-Instruct'

# --- development knobs (tune on TREC 2021 ONLY) ---
DEV_SET   = 'trec21'     # develop here; switch to 'trec22' for the SINGLE final confirmation
RERANK_K  = 30           # how many of the ensemble top-K to CoT-rerank
MAX_NEW   = 256          # CoT reasoning budget
os.environ['CTMATCH_DATA_ROOT'] = DATA_ROOT
FEATURES = ['bm25','bm25_rank','dense','dense_rank','rrf','clf_rel','clf_partial','v2_rel','llm_yesno','llm_scored']
print('config set | DEV_SET =', DEV_SET)

In [ ]:
import json, numpy as np, lightgbm as lgb
from datasets import load_dataset
from ctmatch.evaluation.eval_utils import load_eval_datasets

# corpus text
_idx = load_dataset('semaj83/ctmatch_ir', data_files='index2docid.txt', split='train')
corpus_ids = [r['text'].strip() for r in _idx]
with open(FULLTEXT_CORPUS) as f:
    corpus_txt = [l.rstrip('\n') for l in f]
id2txt = dict(zip(corpus_ids, corpus_txt))

# cached features + meta + frozen ensemble
d = np.load(FEATURE_CACHE, allow_pickle=True)
with open(META_CACHE) as f:
    m = json.load(f)
mtr = [tuple(x) for x in m['mtr']]; mte = [tuple(x) for x in m['mte']]
booster = lgb.Booster(model_file=ENSEMBLE_MODEL)

# rebuild the LLM feature column exactly as train_ensemble_full does (floor + indicator),
# so the frozen booster sees identical inputs
llm_lookup = {}
with open(f'{DATA_ROOT}/llm_reranker_scores.jsonl') as f:
    for line in f:
        r = json.loads(line); llm_lookup[(r['source'], r['topic_id'], r['doc_id'])] = r['llm_score']
FLOOR = min(llm_lookup.values()) - 5.0
def with_llm(X, meta):
    raw = np.array([llm_lookup.get(tuple(mm), np.nan) for mm in meta], dtype=np.float32)
    val = np.where(np.isnan(raw), FLOOR, raw).astype(np.float32)
    return np.column_stack([X[:, :8], val, (~np.isnan(raw)).astype(np.float32)]).astype(np.float32)

split = {'trec21': (d['Xtr'], mtr), 'kz': (d['Xtr'], mtr), 'trec22': (d['Xte'], mte)}
all_sets = load_eval_datasets(TREC_ROOT, KZ_ROOT)
print('loaded frozen ensemble + features; DEV_SET =', DEV_SET)

In [ ]:
# Build the frozen-ensemble ranking for DEV_SET, then take each topic's top-RERANK_K to CoT-judge.
X, meta = split[DEV_SET]
Xl = with_llm(X, meta)
scores = booster.predict(Xl)
rows = [(t, dcid, float(s)) for (n, t, dcid), s in zip(meta, scores) if n == DEV_SET]
from collections import defaultdict
base_run = defaultdict(dict)
for t, dcid, s in rows:
    base_run[t][dcid] = s
topic2text = all_sets[DEV_SET]['topic2text']
rel_dict = all_sets[DEV_SET]['rel_dict']
heads = {t: [d for d, _ in sorted(dd.items(), key=lambda x: x[1], reverse=True)[:RERANK_K]]
         for t, dd in base_run.items()}
print(f'{DEV_SET}: {len(base_run)} topics, CoT-judging top-{RERANK_K} each '
      f'({sum(len(v) for v in heads.values())} pairs)')

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

tok = AutoTokenizer.from_pretrained(LLM_MODEL, padding_side='left')
if tok.pad_token is None: tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL, torch_dtype=torch.float16, device_map='auto').eval()
YES = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ['yes','Yes',' yes',' Yes']})
NO  = sorted({tok.encode(w, add_special_tokens=False)[0] for w in ['no','No',' no',' No']})
SYS = 'You are a meticulous clinical trial eligibility assessor.'

def cot_prompt(patient, trial):
    return tok.apply_chat_template([{'role':'system','content':SYS},{'role':'user','content':(
        f'Patient:\n{patient}\n\nTrial eligibility criteria:\n{trial[:2000]}\n\n'
        'Reason step by step: (1) the key inclusion criteria and whether the patient plausibly meets '
        'them, making reasonable clinical inferences; (2) any exclusion criteria that clearly apply. '
        'Then give a final answer.')}], tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def cot_score(patient, trials, batch=4):
    out = []
    for i in range(0, len(trials), batch):
        chunk = trials[i:i+batch]
        enc = tok([cot_prompt(patient, t) for t in chunk], return_tensors='pt', padding=True,
                  truncation=True, max_length=2048).to(llm.device)
        gen = llm.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.eos_token_id)
        # append an eligibility question and read yes/no logprob after the reasoning
        follow = [tok.decode(g[enc['input_ids'].shape[1]:], skip_special_tokens=True) +
                  '\n\nFinal answer — is the patient eligible for this trial? Answer yes or no: '
                  for g in gen]
        p2 = [cot_prompt(patient, t) + f for t, f in zip(chunk, follow)]
        enc2 = tok(p2, return_tensors='pt', padding=True, truncation=True, max_length=2400).to(llm.device)
        lg = llm(**enc2).logits[:, -1, :].float()
        out.extend((torch.logsumexp(lg[:, YES], -1) - torch.logsumexp(lg[:, NO], -1)).cpu().tolist())
    return out

done = set()
if os.path.exists(COT_OUT):
    for line in open(COT_OUT):
        r = json.loads(line); done.add((r['source'], r['topic_id'], r['doc_id']))
with open(COT_OUT, 'a') as f:
    for t, docs in tqdm(heads.items(), desc=f'CoT {DEV_SET}'):
        todo = [d for d in docs if (DEV_SET, t, d) not in done]
        if not todo: continue
        sc = cot_score(topic2text[t], [id2txt.get(d,'') for d in todo])
        for d, s in zip(todo, sc):
            f.write(json.dumps({'source':DEV_SET,'topic_id':t,'doc_id':d,'cot':float(s)})+'\n')
        f.flush()
print('CoT scores written ->', COT_OUT)

In [ ]:
import pytrec_eval
cot = {}
for line in open(COT_OUT):
    r = json.loads(line)
    if r['source'] == DEV_SET: cot[(r['topic_id'], r['doc_id'])] = r['cot']

def zize(d):
    v = np.array(list(d.values())); s = v.std();
    return {k: ((x-v.mean())/s if s>0 else 0.0) for k, x in d.items()}

qrel = {t: {dc: int(r) for dc, r in dd.items()} for t, dd in rel_dict.items() if t in base_run}
ev = pytrec_eval.RelevanceEvaluator(qrel, {'ndcg_cut.10'})
def ndcg_of(run):
    return np.mean([v['ndcg_cut_10'] for v in ev.evaluate(run).values()])

base_ndcg = ndcg_of({t: dd for t, dd in base_run.items()})
print(f'{DEV_SET} frozen-ensemble NDCG@10: {base_ndcg:.4f}\n')
print('alpha  NDCG@10   (blend of CoT over the top-K with ensemble order)')
for a in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    run = {}
    for t, dd in base_run.items():
        head = heads[t]
        cz = zize({d: cot.get((t, d), 0.0) for d in head})
        ez = zize({d: dd[d] for d in head})
        new = {d: a*cz[d] + (1-a)*ez[d] for d in head}
        base_top = max(dd.values()) + 1.0
        run[t] = {d: base_top + s for d, s in new.items()}   # reranked head on top
        for d, s in dd.items():
            if d not in head: run[t][d] = s - 1e6            # tail keeps ensemble order below
    print(f'{a:.1f}    {ndcg_of(run):.4f}')
print(f'\nDEV protocol: pick best alpha HERE (on {DEV_SET}); then set DEV_SET=trec22 and run once.')